In [ ]:
%pip install langchain-text-splitters

In [ ]:
dbutils.library.restartPython()

In [ ]:
from pyspark.sql import functions as F

SOURCE_TABLE = ("workspace.viktoriia_kalenichenko_lab12.document_pages")

pages_df = spark.table(SOURCE_TABLE)

print(f"Прочитано з Delta: {pages_df.count()} сторінок")

In [ ]:
# Беремо початковий текст із pages_df.
prepared_df = pages_df.withColumn(
    "clean_text",
    F.regexp_replace(F.col("text"), r"\r\n?", "\n")
)

# Прибираємо нульовий технічний символ.
prepared_df = prepared_df.withColumn(
    "clean_text",
    F.regexp_replace("clean_text", "\u0000", "")
)

# Прибираємо окремий номер сторінки на початку.
prepared_df = prepared_df.withColumn(
    "clean_text",
    F.regexp_replace("clean_text", r"^\s*\d+[ \t]*\n", "")
)

# Скорочуємо зайві пробіли, але ЗБЕРІГАЄМО переноси рядків.
prepared_df = prepared_df.withColumn(
    "clean_text",
    F.regexp_replace("clean_text", r"[ \t\u00A0]+", " ")
)

# Прибираємо пробіли біля переносів і надлишкові порожні рядки.
prepared_df = prepared_df.withColumn(
    "clean_text",
    F.trim(
        F.regexp_replace(
            F.regexp_replace("clean_text", r" *\n *", "\n"),
            r"\n{3,}",
            "\n\n"
        )
    )
)

# Залишаємо непорожній текст без двох титульних сторінок.
prepared_df = (
    prepared_df
    .withColumn("clean_char_count", F.length("clean_text"))
    .filter(F.col("clean_char_count") > 0)
    .filter(~F.col("page_number").isin(1, 2))
)

print(f"Сторінок для поділу: {prepared_df.count()}")

sample = (
    prepared_df
    .filter(F.col("page_number") == 20)
    .select("clean_text")
    .first()
)

print(sample["clean_text"][:1500])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
    is_separator_regex=False,
)

# Беремо одну очищену сторінку.
page_text = (
    prepared_df
    .filter(F.col("page_number") == 20)
    .select("clean_text")
    .first()["clean_text"]
)

# Ділимо її готовим інструментом.
test_chunks = splitter.split_text(page_text)

print(f"На сторінці 20 створено фрагментів: {len(test_chunks)}")

for number, chunk in enumerate(test_chunks, start=1):
    print(f"\n===== Фрагмент {number}: {len(chunk)} символів =====\n")
    print(chunk)

In [ ]:
import hashlib

PROFILE = "recursive_chars_1200_overlap_200_v1"
chunk_rows = []

pages = (
    prepared_df
    .select("file_name", "source_sha256", "page_number", "clean_text")
    .orderBy("file_name", "page_number")
    .collect()
)

for page in pages:
    # Використовуємо той самий splitter, що в тесті сторінки 20.
    fragments = splitter.split_text(page["clean_text"])

    for chunk_index, chunk_text in enumerate(fragments, start=1):
        identity = (
            f'{page["source_sha256"]}|{page["page_number"]}|'
            f'{PROFILE}|{chunk_index}|{chunk_text}'
        )
        chunk_id = hashlib.sha256(
            identity.encode("utf-8")
        ).hexdigest()

        chunk_rows.append((
            chunk_id,
            PROFILE,
            page["file_name"],
            page["source_sha256"],
            page["page_number"],
            chunk_index,
            chunk_text,
            len(chunk_text),
        ))

chunks_df = spark.createDataFrame(
    chunk_rows,
    schema="""
        chunk_id STRING,
        chunk_profile STRING,
        file_name STRING,
        source_sha256 STRING,
        page_number INT,
        chunk_index INT,
        chunk_text STRING,
        char_count INT
    """
)

# Перевіряємо результат перед збереженням.
assert len(pages) == 144, "Очікували 144 сторінки"
assert chunks_df.select("page_number").distinct().count() == 144, \
    "Не всі сторінки отримали фрагменти"
assert chunks_df.select("chunk_id").distinct().count() == len(chunk_rows), \
    "Знайдено повторні ідентифікатори"
assert all(0 < row[-1] <= 1200 for row in chunk_rows), \
    "Є порожній або завеликий фрагмент"

print(f"Оброблено сторінок: {len(pages)}")
print(f"Створено фрагментів: {len(chunk_rows)}")
print("Перевірки пройдено")

display(
    chunks_df.select(
        "page_number",
        "chunk_index",
        "char_count",
        F.substring("chunk_text", 1, 150).alias("preview")
    )
    .orderBy("page_number", "chunk_index")
    .limit(10)
)

In [ ]:
CHUNKS_TABLE = (
    "workspace.viktoriia_kalenichenko_lab12.document_chunks"
)

(
    chunks_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(CHUNKS_TABLE)
)

saved_count = spark.table(CHUNKS_TABLE).count()
assert saved_count == len(chunk_rows), \
    "Кількість фрагментів після запису не збігається"

print(f"Збережено {saved_count} фрагментів у {CHUNKS_TABLE}")